# SCMP Population + Socio-economic Statistics
### Corrected WorldPop version (Python / Google Colab)

This notebook reproduces, in Python using the `earthengine-api`, a Google Earth Engine (GEE) JavaScript workflow that computes population and socio-economic statistics for six SCMP catchments in the Benue Trough / Lake Chad Basin region.

**Selected catchments:**
1. Benue-Mada
2. Shemankar-Kastina-Ala
3. Wase-Taraba
4. Hawul-Kilange
5. Yedseram
6. Ngada

**Population years:** 2000, 2005, 2010, 2015, 2020 (WorldPop)

**Land use/land cover reference year:** 2020 (Dynamic World)

**Night-time lights reference year:** 2020 (VIIRS VNP46A2)

**Output:** CSV export to Google Drive.


In [ ]:
# Install / import Earth Engine
!pip install -q earthengine-api

import ee

# Authenticate and initialize (replace with your own GEE-enabled cloud project)
ee.Authenticate()
ee.Initialize(project='ee-samuelcoolsdk')


## 1. Catchments

In [ ]:
scmp = ee.FeatureCollection('projects/ee-samuelcoolsdk/assets/SCMP_SHAPEFILES')

shemankar = ee.FeatureCollection(
    'projects/ee-samuelcoolsdk/assets/Shemankar_Kastina_Ala_Catchment'
)


## 2. Select five SCMP catchments

In [ ]:
selected = scmp.filter(
    ee.Filter.inList('NAME', [
        'Benue-Mada',
        'Wase-Taraba',
        'Hawul-Kilange',
        'Yedseram',
        'Ngada',
    ])
)


## 3. Standardize name

In [ ]:
def standardize_name(f):
    return f.set('CATCHMENT', f.get('NAME'))

selected = selected.map(standardize_name)


## 4. Add Shemankar

In [ ]:
shemankar_feature = ee.Feature(
    shemankar.geometry(),
    {'CATCHMENT': 'Shemankar-Kastina-Ala'},
)


## 5. Merge six catchments

In [ ]:
catchments = selected.merge(ee.FeatureCollection([shemankar_feature]))


## 6. Add catchment area

In [ ]:
def add_area(f):
    area = f.geometry().area().divide(1000000)
    return f.set('AREA_KM2', area)

catchments = catchments.map(add_area)

print('SELECTED CATCHMENTS')
print(catchments.getInfo())


## 7. WorldPop

**Important:** filter spatially *before* doing anything else. This prevents Earth Engine from accumulating all ~5,221 images in the full WorldPop collection.

In [ ]:
worldpop = ee.ImageCollection('WorldPop/GP/100m/pop').filterBounds(
    catchments.geometry()
)


## 8. Get WorldPop mosaic for a year

In [ ]:
def world_pop_year(year):
    return (
        worldpop
        .filter(ee.Filter.eq('year', year))
        .mosaic()
        .select('population')
    )


## 9. Load available years

In [ ]:
pop2000 = world_pop_year(2000)
pop2005 = world_pop_year(2005)
pop2010 = world_pop_year(2010)
pop2015 = world_pop_year(2015)
pop2020 = world_pop_year(2020)


## 10. Display check

In [ ]:
print('WorldPop 2000', pop2000.getInfo())
print('WorldPop 2005', pop2005.getInfo())
print('WorldPop 2010', pop2010.getInfo())
print('WorldPop 2015', pop2015.getInfo())
print('WorldPop 2020', pop2020.getInfo())


## 11. Function to calculate population

In [ ]:
def population_by_catchment(image):
    summed = image.reduceRegions(
        collection=catchments,
        reducer=ee.Reducer.sum(),
        scale=100,
        crs='EPSG:4326',
        tileScale=8,
    )

    def to_pop_feature(f):
        return ee.Feature(None, {
            'CATCHMENT': f.get('CATCHMENT'),
            'POP': f.get('sum'),
        })

    return summed.map(to_pop_feature)


## 12. Extract population

In [ ]:
p2000 = population_by_catchment(pop2000)
p2005 = population_by_catchment(pop2005)
p2010 = population_by_catchment(pop2010)
p2015 = population_by_catchment(pop2015)
p2020 = population_by_catchment(pop2020)


## 13. Create population summary

In [ ]:
def pop_value(fc, name):
    return ee.Number(
        ee.Feature(fc.filter(ee.Filter.eq('CATCHMENT', name)).first()).get('POP')
    )

def build_population_row(c):
    name = c.get('CATCHMENT')

    P2000 = pop_value(p2000, name)
    P2005 = pop_value(p2005, name)
    P2010 = pop_value(p2010, name)
    P2015 = pop_value(p2015, name)
    P2020 = pop_value(p2020, name)

    # Population change
    change = P2020.subtract(P2000)
    growth_percent = change.divide(P2000).multiply(100)

    # Population density
    area = ee.Number(c.get('AREA_KM2'))
    density = P2020.divide(area)

    return c.set({
        'POP_2000': P2000,
        'POP_2005': P2005,
        'POP_2010': P2010,
        'POP_2015': P2015,
        'POP_2020': P2020,
        'POP_CHANGE_2000_2020': change,
        'POP_GROWTH_PERCENT': growth_percent,
        'POP_DENSITY_2020': density,
    })

population_table = catchments.map(build_population_row)


## 14. Dynamic World 2020

In [ ]:
dw = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filterDate('2020-01-01', '2021-01-01')
    .filterBounds(catchments.geometry())
)


## 15. Mode land cover

In [ ]:
dw2020 = dw.select('label').mode()


## 16. Pixel area (km2)

In [ ]:
pixel_area = ee.Image.pixelArea().divide(1000000)


## 17. Built-up (Dynamic World class 6)

In [ ]:
built = pixel_area.updateMask(dw2020.eq(6)).rename('BUILT_KM2')


## 18. Cropland (Dynamic World class 4)

In [ ]:
crops = pixel_area.updateMask(dw2020.eq(4)).rename('CROP_KM2')


## 19. Water (Dynamic World class 0)

In [ ]:
water = pixel_area.updateMask(dw2020.eq(0)).rename('WATER_KM2')


## 20. Vegetation

Trees = 1, Grass = 2, Flooded vegetation = 3, Shrub/Scrub = 5

In [ ]:
vegetation_mask = (
    dw2020.eq(1)
    .Or(dw2020.eq(2))
    .Or(dw2020.eq(3))
    .Or(dw2020.eq(5))
)

vegetation = pixel_area.updateMask(vegetation_mask).rename('VEGETATION_KM2')


## 21. Bare ground (Dynamic World class 7)

In [ ]:
bare = pixel_area.updateMask(dw2020.eq(7)).rename('BARE_KM2')


## 22. LULC stack

In [ ]:
lulc_stack = ee.Image.cat([built, crops, water, vegetation, bare])


## 23. LULC statistics

In [ ]:
lulc_stats = lulc_stack.reduceRegions(
    collection=population_table,
    reducer=ee.Reducer.sum(),
    scale=10,
    crs='EPSG:4326',
    tileScale=8,
)


## 24. LULC percentages

In [ ]:
def add_lulc_percentages(f):
    area = ee.Number(f.get('AREA_KM2'))

    return f.set({
        'BUILT_PERCENT': ee.Number(f.get('BUILT_KM2')).divide(area).multiply(100),
        'CROP_PERCENT': ee.Number(f.get('CROP_KM2')).divide(area).multiply(100),
        'VEGETATION_PERCENT': ee.Number(f.get('VEGETATION_KM2')).divide(area).multiply(100),
        'WATER_PERCENT': ee.Number(f.get('WATER_KM2')).divide(area).multiply(100),
        'BARE_PERCENT': ee.Number(f.get('BARE_KM2')).divide(area).multiply(100),
    })

lulc_stats = lulc_stats.map(add_lulc_percentages)


## 25. VIIRS night-time light

In [ ]:
viirs = (
    ee.ImageCollection('NASA/VIIRS/002/VNP46A2')
    .filterDate('2020-01-01', '2021-01-01')
    .filterBounds(catchments.geometry())
)


## 26. Night-time light composite

In [ ]:
ntl = viirs.select('Gap_Filled_DNB_BRDF_Corrected_NTL').mean()


## 27. Night-time light statistics

In [ ]:
ntl_reducer = (
    ee.Reducer.mean()
    .combine(reducer2=ee.Reducer.max(), sharedInputs=True)
    .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True)
)

ntl_stats = ntl.reduceRegions(
    collection=lulc_stats,
    reducer=ntl_reducer,
    scale=500,
    crs='EPSG:4326',
    tileScale=8,
)


## 28. Rename NTL variables

In [ ]:
def rename_ntl(f):
    return f.set({
        'NTL_MEAN': f.get('mean'),
        'NTL_MAX': f.get('max'),
        'NTL_STDDEV': f.get('stdDev'),
    })

final_table = ntl_stats.map(rename_ntl)


## 29. Final columns

In [ ]:
final_table = final_table.select([
    'CATCHMENT',
    'AREA_KM2',

    # Population
    'POP_2000',
    'POP_2005',
    'POP_2010',
    'POP_2015',
    'POP_2020',

    'POP_CHANGE_2000_2020',
    'POP_GROWTH_PERCENT',
    'POP_DENSITY_2020',

    # Land use
    'BUILT_KM2',
    'BUILT_PERCENT',

    'CROP_KM2',
    'CROP_PERCENT',

    'VEGETATION_KM2',
    'VEGETATION_PERCENT',

    'WATER_KM2',
    'WATER_PERCENT',

    'BARE_KM2',
    'BARE_PERCENT',

    # Night lights
    'NTL_MEAN',
    'NTL_MAX',
    'NTL_STDDEV',
])


## 30. Final output

In [ ]:
print('FINAL POPULATION AND SOCIO-ECONOMIC STATISTICS')
print(final_table.getInfo())


## 31. Export CSV

In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=final_table,
    description='SCMP_Population_Socioeconomic_Statistics_2000_2020',
    folder='SCMP_Climate_Socioeconomic',
    fileNamePrefix='SCMP_Population_Socioeconomic_Statistics_2000_2020',
    fileFormat='CSV',
)

task.start()
print('Export task started:', task.id)
